# D2-02 Edges fundamentals and regionalisation concepts

In this notebook, we do **not** use `edges` directly yet. Instead, we build the conceptual bridge from Day 1 by comparing a conventional characterization step with an exchange-based one.


## Learning goals

After this notebook, you should be able to:

- explain how a conventional LCIA step uses one CF per biosphere flow
- explain how an exchange-based LCIA step can use different CFs for different exchanges of the same flow
- relate that difference to locations and other edge attributes
- connect the manual matrix formulation to the motivation behind `edges`


## Background references

- Sacchi, R., Menacho, A. H., Seitfudem, G., Agez, M., Schlesinger-Martinat, J., Koyamparambath, A., Saldivar, J. S., Loubet, P., & Bauer, C. (2025). Contextual LCIA without the overhead: an exchange-based framework for flexible impact assessment. *The International Journal of Life Cycle Assessment, 30*(12), 3087-3101. https://doi.org/10.1007/s11367-025-02551-7
- Heijungs, R., & Suh, S. (2002). *The Computational Structure of Life Cycle Assessment*. Kluwer Academic Publishers. https://doi.org/10.1007/978-94-015-9900-9


## 1) Two ways to write the characterization step

In the `brightway` matrix formulation, one biosphere flow gets one characterization factor:

$$
H = QG
$$

where $Q = \operatorname{diag}(q)$ is diagonal.

In an exchange-based formulation, we proceed to a Hadamard multiplication, where characterization factors can depend on the **supplier** and the **consumer** activities at the same time:

$$
H^{\text{B}} = E^{\text{B}} \cdot G
$$

where $E^{\text{B}}$ has the same shape as $G$, and each entry can depend on edges' **consumer** or **supplier** attributes such as name, location, etc.


In [1]:
import numpy as np
import pandas as pd

## 2) Rebuild the compact regionalized CAM system from Day 1

We reuse the same small supply chain from `D1-03` so that only the characterization logic changes.


In [2]:
products = [
    'Li+ (aq.) [kg]',
    'Electricity (CL) [kWh]',
    'Li2CO3 [kg]',
    'Electricity (CN) [kWh]',
    'CAM [kg]',
    'Electricity (DE) [kWh]',
]
activities = [
    'Li+ extraction (CL)',
    'Hydropower (CL)',
    'Li2CO3 production (CN)',
    'Hydropower (CN)',
    'CAM production (DE)',
    'Hydropower (DE)',
]
flows = ['Water, underground [kg]', 'Water, surface [kg]', 'Water, unspecified [kg]']

A = np.array([
    [ 1.0,  0.0, -0.8, 0.0, -0.1, 0.0],
    [-0.5,  1.0,  0.0, 0.0,-25.0, 0.0],
    [ 0.0,  0.0,  1.0, 0.0, -0.2, 0.0],
    [ 0.0,  0.0, -3.0, 1.0,  0.0, 0.0],
    [ 0.0,  0.0,  0.0, 0.0,  1.0, 0.0],
    [ 0.0,  0.0,  0.0, 0.0,-25.0, 1.0],
])

B = np.array([
    [2.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 3.0, 0.0, 0.0, 0.0],
    [0.0, 0.02, 0.0, 0.015, 0.2, 0.005],
])

f = np.array([0.0, 0.0, 0.0, 0.0, 1.0, 0.0])
q_global = np.array([39.5, 39.5, 39.5])

In [3]:
pd.DataFrame(A, index=products, columns=activities)

,Li+ extraction (CL),Hydropower (CL),Li2CO3 production (CN),Hydropower (CN),CAM production (DE),Hydropower (DE)
Li+ (aq.) [kg],1.0,0.0,-0.8,0.0,-0.1,0.0
Electricity (CL) [kWh],-0.5,1.0,0.0,0.0,-25.0,0.0
Li2CO3 [kg],0.0,0.0,1.0,0.0,-0.2,0.0
Electricity (CN) [kWh],0.0,0.0,-3.0,1.0,0.0,0.0
CAM [kg],0.0,0.0,0.0,0.0,1.0,0.0
Electricity (DE) [kWh],0.0,0.0,0.0,0.0,-25.0,1.0


In [4]:
pd.DataFrame(B, index=flows, columns=activities)

,Li+ extraction (CL),Hydropower (CL),Li2CO3 production (CN),Hydropower (CN),CAM production (DE),Hydropower (DE)
"Water, underground [kg]",2.0,0.00,0.0,0.000,0.0,0.000
"Water, surface [kg]",0.0,0.00,3.0,0.000,0.0,0.000
"Water, unspecified [kg]",0.0,0.02,0.0,0.015,0.2,0.005


In [5]:
pd.DataFrame(f, index=products, columns=['Demand'])

,Demand
Li+ (aq.) [kg],0.0
Electricity (CL) [kWh],0.0
Li2CO3 [kg],0.0
Electricity (CN) [kWh],0.0
CAM [kg],1.0
Electricity (DE) [kWh],0.0


In [6]:
pd.DataFrame(q_global, index=flows, columns=['Global CF'])

,Global CF
"Water, underground [kg]",39.5
"Water, surface [kg]",39.5
"Water, unspecified [kg]",39.5


## 3) Conventional characterization with one global factor per flow

This reproduces the Day 1 logic: the same water-withdrawal factor is applied everywhere.


In [7]:
s = np.linalg.solve(A, f)
G = B @ np.diag(s)
Q_global = np.diag(q_global)
H_global = Q_global @ G

In [8]:
print('Scaling vector s')
pd.DataFrame(s, index=activities, columns=['Scale'])

Scaling vector s


,Scale
Li+ extraction (CL),0.26
Hydropower (CL),25.13
Li2CO3 production (CN),0.20
Hydropower (CN),0.60
CAM production (DE),1.00
Hydropower (DE),25.00


In [9]:
print('Inventory matrix G')
pd.DataFrame(G, index=flows, columns=activities)

Inventory matrix G


,Li+ extraction (CL),Hydropower (CL),Li2CO3 production (CN),Hydropower (CN),CAM production (DE),Hydropower (DE)
"Water, underground [kg]",0.52,0.0000,0.0,0.000,0.0,0.000
"Water, surface [kg]",0.00,0.0000,0.6,0.000,0.0,0.000
"Water, unspecified [kg]",0.00,0.5026,0.0,0.009,0.2,0.125


In [10]:
print('Characterized inventory with global CFs')
pd.DataFrame(H_global, index=flows, columns=activities)

Characterized inventory with global CFs


,Li+ extraction (CL),Hydropower (CL),Li2CO3 production (CN),Hydropower (CN),CAM production (DE),Hydropower (DE)
"Water, underground [kg]",20.54,0.0000,0.0,0.0000,0.0,0.0000
"Water, surface [kg]",0.00,0.0000,23.7,0.0000,0.0,0.0000
"Water, unspecified [kg]",0.00,19.8527,0.0,0.3555,7.9,4.9375


In [11]:
print(f'Total score with global CFs: {H_global.sum():.1f}') # let's display only one decimal

Total score with global CFs: 77.3


## 4) Exchange-based characterization with location-specific AWARE factors

We now mirror the supplementary CAM example more closely.

The global case used one factor everywhere: `39.5`.

In the exchange-based case, the factor depends on the location of the activity where the water withdrawal occurs (the **consumer**), according to `AWARE 2.0`:

- Chile (`CL`): `45.5`
- China (`CN`): `6.3`
- Germany (`DE`): `2.1`

So $E^B$ has the same shape as `G`, but its non-zero entries now depend on the activity location attached to each exchange.


In [12]:
G_df = pd.DataFrame(G, index=flows, columns=activities)

activity_locations = pd.Series(
    {
        'Li+ extraction (CL)': 'CL',
        'Hydropower (CL)': 'CL',
        'Li2CO3 production (CN)': 'CN',
        'Hydropower (CN)': 'CN',
        'CAM production (DE)': 'DE',
        'Hydropower (DE)': 'DE',
    },
    name='Location',
)
aware_by_location = {'CL': 45.5, 'CN': 6.3, 'DE': 2.1}

location_factors = pd.DataFrame(
    {
        'Location': ['CL', 'CN', 'DE'],
        'AWARE 2.0 CF': [45.5, 6.3, 2.1],
    }
)

In [13]:
location_factors

,Location,AWARE 2.0 CF
0,CL,45.5
1,CN,6.3
2,DE,2.1


In [14]:
activity_locations.to_frame()


,Location
Li+ extraction (CL),CL
Hydropower (CL),CL
Li2CO3 production (CN),CN
Hydropower (CN),CN
CAM production (DE),DE
Hydropower (DE),DE


In [15]:
E_B = pd.DataFrame(0.0, index=flows, columns=activities)

for activity, location in activity_locations.items():
    E_B.loc[G_df[activity] != 0.0, activity] = aware_by_location[location]

E_B

,Li+ extraction (CL),Hydropower (CL),Li2CO3 production (CN),Hydropower (CN),CAM production (DE),Hydropower (DE)
"Water, underground [kg]",45.5,0.0,0.0,0.0,0.0,0.0
"Water, surface [kg]",0.0,0.0,6.3,0.0,0.0,0.0
"Water, unspecified [kg]",0.0,45.5,0.0,6.3,2.1,2.1


In [16]:
H_edges = E_B * G_df

In [17]:
print('Exchange-based characterization matrix E^B')
display(E_B)

Exchange-based characterization matrix E^B


,Li+ extraction (CL),Hydropower (CL),Li2CO3 production (CN),Hydropower (CN),CAM production (DE),Hydropower (DE)
"Water, underground [kg]",45.5,0.0,0.0,0.0,0.0,0.0
"Water, surface [kg]",0.0,0.0,6.3,0.0,0.0,0.0
"Water, unspecified [kg]",0.0,45.5,0.0,6.3,2.1,2.1


In [18]:
print('Characterized inventory with exchange-based CFs')
display(H_edges)

Characterized inventory with exchange-based CFs


,Li+ extraction (CL),Hydropower (CL),Li2CO3 production (CN),Hydropower (CN),CAM production (DE),Hydropower (DE)
"Water, underground [kg]",23.66,0.0000,0.00,0.0000,0.00,0.0000
"Water, surface [kg]",0.00,0.0000,3.78,0.0000,0.00,0.0000
"Water, unspecified [kg]",0.00,22.8683,0.00,0.0567,0.42,0.2625


In [19]:
print(f'Total score with exchange-based CFs: {H_edges.sum().sum():.2f}.')

Total score with exchange-based CFs: 51.05.


Do you notice the difference in score? If so, why is that?

## Checkpoint 1

Compare the two totals and answer:

- Which activity columns change between the conventional and exchange-based results?
- Why can the same biosphere flow, such as `Water, unspecified [kg]`, receive different factors in different columns?


In [ ]:
# TODO
# global_total = ...
# edges_total = ...
# changed_columns = ...


In [ ]:
global_total = float(H_global.sum())
changed = H_edges - pd.DataFrame(H_global, index=flows, columns=activities)
changed_columns = changed.sum(axis=0)
changed_columns = changed_columns[changed_columns != 0].sort_values(ascending=False)

print('Global total:', global_total)
print('Exchange-based total:', float(H_edges.to_numpy().sum()))
print('Changed columns:')
display(changed_columns.to_frame('Delta score'))
print('Explanation: the factor is attached to each flow-activity exchange, so the same flow can receive different CFs depending on where the consuming activity is located.')


## 5) Why location lives on the edge

In this CAM example, the biosphere flows are still the same three water-withdrawal flows, but their characterization differs because the withdrawals happen in different activity locations.

That is why `edges` uses edge attributes from both ends of the exchange. The characterization factor can depend on:

- the supplier flow identity
- the consumer activity identity
- the geography attached to the consumer or supplier
- other activity metadata used in the method definition


In [ ]:
edge_attributes = (
    G_df.stack()
    .rename('Inventory amount')
    .reset_index()
    .rename(columns={'level_0': 'supplier', 'level_1': 'consumer'})
)
edge_attributes = edge_attributes[edge_attributes['Inventory amount'] != 0].copy()
edge_attributes['consumer location'] = edge_attributes['consumer'].map(activity_locations)
edge_attributes['CF used'] = [E_B.loc[row['supplier'], row['consumer']] for _, row in edge_attributes.iterrows()]
edge_attributes


## Recap

After this notebook, you should now be able to:

- explain why a diagonal characterization matrix is limited for regionalized LCIA
- construct a simple exchange-based CF matrix by hand from location-specific factors
- compare `Q @ G` with an element-wise `E^B * G` formulation
- explain why location is an edge attribute in this context
